# Sessão 1 – Inicialização do Chat (Foundry Local)

Este notebook inicializa o Foundry Local, faz o download do alias do modelo preferido e realiza tanto uma conclusão de chat padrão quanto uma conclusão de chat em streaming.


# Cenário
Esta sessão apresenta o mínimo necessário para fazer um pequeno modelo de linguagem local responder através do Foundry Local. Você irá:
- Instalar o SDK / dependências do cliente.
- Inicializar o gerenciador Foundry Local para um alias escolhido (padrão: `phi-4-mini`).
- Aplicar um monkey patch defensivo para lidar com campos opcionais nos metadados do modelo.
- Enviar uma solicitação padrão de conclusão de chat.
- Transmitir uma resposta token por token.

O objetivo é validar o seu runtime local e o caminho de rede antes de avançar para RAG, roteamento ou agentes.


### Explicação: Instalação de Dependências
Instala os pacotes Python necessários para este fluxo de chat minimalista:
- `foundry-local-sdk`: Gerir modelos locais e o ciclo de vida dos serviços.
- `openai`: Abstração de cliente familiar para completions de chat.
- `rich`: Impressão formatada para uma saída mais clara em notebooks.

Reexecutar é seguro (idempotente). Ignore se o seu ambiente já tiver estes pacotes.


In [1]:
# Install required libraries (idempotent)
%pip install -q foundry-local-sdk openai rich


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Explicação: Importações Principais
Traz módulos utilizados ao longo do notebook:
- `FoundryLocalManager` para interagir com o runtime local do modelo.
- Cliente `OpenAI` para podermos reutilizar a interface familiar da API de conclusão de chat.
- `rich.print` para saída formatada.

Nenhuma chamada de rede ocorre aqui—isto apenas prepara o namespace.


In [2]:
import os
from foundry_local import FoundryLocalManager
from foundry_local.models import FoundryModelInfo
from openai import OpenAI
from rich import print

### Explicação: Inicialização do Gestor & Patch de Metadados
Inicializa o `FoundryLocalManager` para o alias escolhido e aplica um monkey-patch defensivo para lidar de forma elegante com respostas de serviço onde `promptTemplate` pode ser `null`.

Resultados principais:
- Confirma o estado do serviço e o endpoint.
- Lista os modelos em cache (verifica o armazenamento local).
- Resolve o ID concreto do modelo para o alias (utilizado em chamadas de chat posteriores).

Se encontrar problemas de validação nos metadados brutos do serviço, este padrão mostra como sanitizar sem precisar modificar o SDK.


In [3]:
# Monkeypatch to tolerate service responses where promptTemplate is null
_original_from_list_response = FoundryModelInfo.from_list_response

def _safe_from_list_response(response):  # type: ignore
    try:
        if isinstance(response, dict) and response.get("promptTemplate") is None:
            # Normalize to empty dict so pydantic validation passes
            response["promptTemplate"] = {}
    except Exception as e:  # pragma: no cover
        print(f"[yellow]Warning: safe wrapper encountered issue normalizing promptTemplate: {e}[/yellow]")
    return _original_from_list_response(response)

# Apply patch only once
if getattr(FoundryModelInfo.from_list_response, "__name__", "") != "_safe_from_list_response":
    FoundryModelInfo.from_list_response = staticmethod(_safe_from_list_response)  # type: ignore

ALIAS = os.getenv('FOUNDRY_LOCAL_ALIAS', 'phi-4-mini')
manager = FoundryLocalManager(ALIAS)
print(f'[bold green]Service running:[/bold green] {manager.is_service_running()}')
print(f'Endpoint: {manager.endpoint}')
print('Cached models:', manager.list_cached_models())
model_id = manager.get_model_info(ALIAS).id
print(f'Using model id: {model_id}')

Service running: True

Endpoint: http://127.0.0.1:50262/v1

Cached models:
[
    FoundryModelInfo(
        alias='phi-4-mini',
        id='Phi-4-mini-instruct-generic-gpu:4',
        version='4',
        execution_provider='WebGpuExecutionProvider',
        device_type=<DeviceType.GPU: 'GPU'>,
        uri='azureml://registries/azureml/models/Phi-4-mini-instruct-generic-gpu/versions/4',
        file_size_mb=3809,
        prompt_template={
            'system': '<|system|>{Content}<|end|>',
            'user': '<|user|>{Content}<|end|>',
            'assistant': '<|assistant|>{Content}<|end|>',
            'prompt': '<|user|>{Content}<|end|><|assistant|>'
        },
        provider='AzureFoundry',
        publisher='Microsoft',
        license='MIT',
        task='chat-completion',
        ep_override=None
    ),
    FoundryModelInfo(
        alias='qwen2.5-0.5b',
        id='qwen2.5-0.5b-instruct-generic-gpu:3',
        version='3',
        execution_provider='WebGpuExecutionProvider',
        device_type=<DeviceType.GPU: 'GPU'>,
        uri='azureml://registries/azureml/models/qwen2.5-0.5b-instruct-generic-gpu/versions/3',
        file_size_mb=700,
        prompt_template={
            'system': '<|im_start|>system\n{Content}<|im_end|>',
            'user': '<|im_start|>user\n{Content}<|im_end|>',
            'assistant': '<|im_start|>assistant\n{Content}<|im_end|>',
            'prompt': '<|im_start|>user\n{Content}<|im_end|>\n<|im_start|>assistant'
        },
        provider='AzureFoundry',
        publisher='Microsoft',
        license='apache-2.0',
        task='chat-completion',
        ep_override=None
    ),
    FoundryModelInfo(
        alias='phi-3.5-mini',
        id='Phi-3.5-mini-instruct-generic-gpu:1',
        version='1',
        execution_provider='WebGpuExecutionProvider',
        device_type=<DeviceType.GPU: 'GPU'>,
        uri='azureml://registries/azureml/models/Phi-3.5-mini-instruct-generic-gpu/versions/1',
        file_size_mb=2211,
        prompt_template={
            'prompt': '<|user|>\n{Content}<|end|>\n<|assistant|>',
            'assistant': '<|assistant|>\n{Content}<|end|>'
        },
        provider='AzureFoundry',
        publisher='Microsoft',
        license='MIT',
        task='chat-completion',
        ep_override=None
    )
]

Using model id: Phi-4-mini-instruct-generic-gpu:4

### Explicação: Conclusão Básica de Chat
Cria um cliente compatível com `OpenAI` apontando para o endpoint local do Foundry e realiza uma única conclusão de chat não contínua. Foco aqui:
- Garantir que o modelo responda sem erros.
- Validar a latência / formato de saída.
- Manter `max_tokens` modesto para conservar recursos.

Se isto falhar, verifique novamente se o serviço Foundry Local está em execução e se o alias está a resolver corretamente.


In [4]:
client = OpenAI(base_url=manager.endpoint, api_key=manager.api_key or 'not-needed')
prompt = 'List two benefits of local inference for privacy.'
resp = client.chat.completions.create(
    model=model_id,
    messages=[{'role':'user','content':prompt}],
    max_tokens=120,
    temperature=0.5
)
print(resp.choices[0].message.content)

Local inference for privacy refers to the practice of performing data analysis on a local device without sending 
sensitive information to a central server. Two benefits of this approach are:


1. **Enhanced Privacy**: Local inference keeps personal data on the user's device, reducing the risk of data 
breaches and unauthorized access. Since the data is not transmitted over the network, it is less susceptible to 
interception by malicious actors.


2. **Data Sovereignty**: Users retain control over their data, as it does not leave their device. This means that 
individuals or organizations can comply with local data protection regulations, such as the General

### Explicação: Conclusão de Chat em Streaming
Demonstra o streaming de tokens para melhorar a latência percebida e a experiência interativa do utilizador. O loop imprime deltas incrementais à medida que chegam:
- Útil para interfaces de chat onde a saída parcial inicial é importante.
- Permite medir o fluxo de tokens em comparação com a latência de conclusão total.

Pode adaptar este padrão para acumular tokens, atualizar um widget de progresso ou abortar a geração a meio.


In [5]:
# Streaming example
stream = client.chat.completions.create(
    model=model_id,
    messages=[{'role':'user','content':'Give a one-sentence definition of edge AI.'}],
    stream=True,
    max_tokens=60,
    temperature=0.4
)
for chunk in stream:
    delta = chunk.choices[0].delta
    if delta and delta.content:
        print(delta.content, end='', flush=True)
print()

Edge

AI

refers

to

artificial

...

latency

and

bandwidth

usage

.

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso**:  
Este documento foi traduzido utilizando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autoritária. Para informações críticas, recomenda-se uma tradução profissional realizada por humanos. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
